In [30]:
import numpy as np
import pandas as pd
import sklearn as sk
import yfinance as yf

In [31]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.model_selection import cross_val_predict,train_test_split

In [32]:
from sklearn.ensemble import HistGradientBoostingRegressor

In [33]:
model_quant=HistGradientBoostingRegressor(random_state=42)
model_cat=RandomForestRegressor(n_estimators=50,random_state=42)
model_embed=SVR(kernel='rbf')

Generate meta features!

In [34]:
X_train=pd.read_csv("../data/processed/X_train.csv")
Y_train=pd.read_csv("../data/processed/Y_train.csv")
X_test=pd.read_csv("../data/processed/X_test.csv")
Y_test=pd.read_csv("../data/processed/Y_test.csv")

In [35]:
X_train.head()

,symbol,trailingPE,forwardPE,averageVolume,marketCap,enterpriseValue,profitMargins,bookValue,priceToBook,earningsQuarterlyGrowth,...,dim_374,dim_375,dim_376,dim_377,dim_378,dim_379,dim_380,dim_381,dim_382,dim_383
0,LOW,19.496202,16.906004,2865616.0,1.295966e+11,1.730865e+11,0.07712,-17.677,-13.069526,-0.110,...,-0.013460,0.012418,0.007428,-0.063844,0.009686,0.025997,-0.071902,-0.044696,0.031740,0.054001
1,SRE,36.072727,17.941145,3972883.0,6.481059e+10,1.102028e+11,0.13400,48.403,2.049460,-0.479,...,0.110943,-0.084352,-0.085241,-0.049469,0.045237,0.052692,-0.083848,0.040969,-0.059888,-0.047014
2,ALGN,30.194690,13.881318,1255170.0,1.224055e+10,1.118027e+10,0.10170,56.739,3.006750,0.308,...,-0.060751,0.036934,0.054971,-0.048341,0.058140,0.071680,-0.071393,-0.087730,0.009890,-0.060807
3,AME,34.107810,25.035755,1401932.0,5.000269e+10,5.209699e+10,0.19999,46.406,4.703918,0.029,...,-0.003118,0.050149,-0.032082,-0.005393,0.017916,0.005058,0.035698,-0.069102,0.007981,-0.000538
4,TYL,46.948612,23.822426,658453.0,1.453033e+10,1.407574e+10,0.13532,85.870,3.936532,0.005,...,0.009151,-0.024399,0.062967,-0.106427,0.013298,0.056442,-0.022533,-0.041416,0.052927,0.009217


In [36]:
quant_columns = [
    'trailingPE', 'forwardPE', 'averageVolume', 'marketCap', 
    'enterpriseValue', 'profitMargins', 'bookValue', 
    'priceToBook', 'earningsQuarterlyGrowth'
]

In [37]:
X_quant_train = X_train[quant_columns]
X_quant_test = X_test[quant_columns]


In [38]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_quant_train = scaler.fit_transform(X_quant_train)

In [39]:
embed_columns = [col for col in X_train.columns if col.startswith('dim_')]
X_embed_train = X_train[embed_columns]
X_embed_test = X_test[embed_columns]

In [40]:
cat_key_columns=["industryKey","sector","country"]
cat_columns = [
    col for col in X_train.columns 
    if any(keyword in col for keyword in cat_key_columns)
]
X_cat_train=X_train[cat_columns]
X_cat_test=X_test[cat_columns]

In [41]:
Y_train

,symbol,enterpriseToEbitda
0,LOW,13.959
1,SRE,20.033
2,ALGN,13.132
3,AME,22.090
4,TYL,32.374
...,...,...
308,RMD,15.840
309,HD,15.388
310,ADBE,10.294
311,MA,21.856


In [42]:
Y_train_series = Y_train['enterpriseToEbitda']
Y_train_series

0      13.959
1      20.033
2      13.132
3      22.090
4      32.374
        ...  
308    15.840
309    15.388
310    10.294
311    21.856
312    34.148
Name: enterpriseToEbitda, Length: 313, dtype: float64

In [43]:
pred_quant = cross_val_predict(model_quant, X_quant_train, Y_train_series, cv=5)
pred_embed = cross_val_predict(model_embed, X_embed_train, Y_train_series, cv=5)
pred_cat=cross_val_predict(model_cat,X_cat_train,Y_train_series,cv=5)

In [44]:
X_meta_train = np.column_stack((pred_quant, pred_cat,pred_embed))

In [45]:
model_quant.fit(X_quant_train, Y_train_series)
model_embed.fit(X_embed_train, Y_train_series)
model_cat.fit(X_cat_train,Y_train_series)
pred_quant_test = model_quant.predict(X_quant_test)
pred_embed_test = model_embed.predict(X_embed_test)
pred_cat_test=model_cat.predict(X_cat_test)
X_meta_test = np.column_stack((pred_quant_test, pred_cat_test,pred_embed_test))

/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but HistGradientBoostingRegressor was fitted without feature names
  warnings.warn(


In [46]:
from sklearn.linear_model import Lasso

In [47]:
judge_model = Lasso(alpha=0.5, positive=True)
judge_model.fit(X_meta_train,Y_train_series)
y_final_pred=judge_model.predict(X_meta_test)

In [48]:
from sklearn.metrics import r2_score, mean_squared_error

In [49]:
Y_test_series=Y_test["enterpriseToEbitda"]

In [50]:
test_r2=r2_score(Y_test_series,y_final_pred)
test_mse=mean_squared_error(Y_test_series,y_final_pred)

In [51]:
test_r2

-9.166124694064933

In [52]:
test_mse

1200.855218213951

In [53]:
quant_r2 = r2_score(Y_train_series, X_meta_train[:, 0])
cat_r2 = r2_score(Y_train_series, X_meta_train[:, 1])
embed_r2 = r2_score(Y_train_series, X_meta_train[:, 2])

print("--- Training Set R2 for Each Expert ---")
print(f"Quant Expert R2: {quant_r2:.4f}")
print(f"Cat Expert R2:   {cat_r2:.4f}")
print(f"Embed Expert R2: {embed_r2:.4f}")

--- Training Set R2 for Each Expert ---
Quant Expert R2: -0.0859
Cat Expert R2:   -0.1408
Embed Expert R2: -0.0036
